## Сбор данных о вратарях во время исполнения пенальти

Импортируем нужные библиотеки для работы с файловой системой, API и таблицами

In [2]:
import os
import pandas as pd
import soccerdata as sd
import warnings
import itertools
from tqdm.auto import tqdm
import glob
import numpy as np
import shutil
import time
import sys

warnings.filterwarnings('ignore')

In [3]:
chunks_dir_espn = os.path.join("..", "data", "gk_espn_chunks")

In [10]:
target_seasons = list(range(2016, 2026))
target_leagues = ['ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'GER-Bundesliga', 'ITA-Serie A']
tasks = list(itertools.product(target_leagues, target_seasons))

In [15]:
os.makedirs(chunks_dir_espn, exist_ok=True)

espn_parser = sd.ESPN(leagues=target_leagues, seasons=target_seasons)

for league in tqdm(target_leagues, mininterval=0.1):
    for season in tqdm(target_seasons, mininterval=0.1):
        chunk_filename = f"gk_espn_{league}_{season}.csv"
        chunk_path = os.path.join(chunks_dir_espn, chunk_filename)

        if os.path.exists(chunk_path):
            continue

        try:
            single_parser = sd.ESPN(leagues=[league], seasons=[season])
            df_step = single_parser.read_lineup().reset_index()

            if df_step is not None and not df_step.empty:
                df_gk = df_step[df_step['position'] == "Goalkeeper"].copy()
                if not df_gk.empty:
                    df_gk.to_csv(chunk_path, index=False)

            time.sleep(1.0)

        except Exception as e:
            tqdm.write(f"Пропущено {league} {season}: {e}")
            continue 

[06/10/26 00:53:16] INFO     Saving cached data to C:\Users\OL\soccerdata\data\ESPN                  ]8;id=2966548;file://C:\Users\OL\AppData\Local\Programs\Python\Python312\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2966549;file://C:\Users\OL\AppData\Local\Programs\Python\Python312\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [16]:
# Ищем все CSV файлы в нашей папке
all_files = glob.glob(os.path.join(chunks_dir_espn, "*.csv"))
df_raw = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

df_raw

,league,season,game,team,player,is_home,position,formation_place,sub_in,sub_out,...,sub_ins,yellow_cards,goals_conceded,saves,shots_faced,goal_assists,shots_on_target,total_goals,total_shots,offsides
0,ENG-Premier League,1617,2016-08-13 Burnley-Swansea City,Burnley,Tom Heaton,True,Goalkeeper,1.0,start,end,...,0.0,0.0,1.0,7.0,0.0,0.0,0.0,0.0,0.0,NaN
1,ENG-Premier League,1617,2016-08-13 Burnley-Swansea City,Swansea City,Lukasz Fabianski,False,Goalkeeper,1.0,start,end,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,NaN
2,ENG-Premier League,1617,2016-08-13 Crystal Palace-West Bromwich Albion,Crystal Palace,Wayne Hennessey,True,Goalkeeper,1.0,start,end,...,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,NaN
3,ENG-Premier League,1617,2016-08-13 Crystal Palace-West Bromwich Albion,West Bromwich Albion,Ben Foster,False,Goalkeeper,1.0,start,end,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,NaN
4,ENG-Premier League,1617,2016-08-13 Everton-Tottenham Hotspur,Everton,Maarten Stekelenburg,True,Goalkeeper,1.0,start,end,...,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36085,ITA-Serie A,2526,2026-05-24 Napoli-Udinese,Udinese,Maduka Okoye,False,Goalkeeper,1.0,start,end,...,0.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,NaN
36086,ITA-Serie A,2526,2026-05-24 Parma-Sassuolo,Parma,Edoardo Corvi,True,Goalkeeper,1.0,start,end,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN
36087,ITA-Serie A,2526,2026-05-24 Parma-Sassuolo,Sassuolo,Stefano Turati,False,Goalkeeper,1.0,start,83,...,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,NaN
36088,ITA-Serie A,2526,2026-05-24 Torino-Juventus,Juventus,Mattia Perin,False,Goalkeeper,1.0,start,end,...,0.0,0.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0,NaN


In [17]:
df_raw.columns

Index(['league', 'season', 'game', 'team', 'player', 'is_home', 'position',
       'formation_place', 'sub_in', 'sub_out', 'appearances',
       'fouls_committed', 'fouls_suffered', 'own_goals', 'red_cards',
       'sub_ins', 'yellow_cards', 'goals_conceded', 'saves', 'shots_faced',
       'goal_assists', 'shots_on_target', 'total_goals', 'total_shots',
       'offsides'],
      dtype='object')

In [18]:
dir_to_save = os.path.join("..", "datasets")
file_name = os.path.join(dir_to_save, "goalkeepers_in_games.csv")
df_raw.to_csv(file_name, index=False)